In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_22076/1857937564.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [34]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing.processing.data.production.internal",

         
#         host="presto.processing.yoda.run",
        # host="processing-.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='sameer.kumar@rapido.bike')

In [28]:
q="""
    with calling_data as(
        select yyyymmdd,count(distinct profile_identity) totalDistinctCalledCaps,count(profile_identity) totalCalls
        from  clevertap.captain_call_support_immutable
        where yyyymmdd between '20250206' and '20250223'
        group by 1
    ),
    rating_data as(
    select yyyymmdd,count(distinct profile_identity) as totalDistinctRatedCaps,count(profile_identity) as totalRatingGiven,
    count(case when event_props_csat='Satisfied' then profile_identity end) as Satisfied,
    count(case when event_props_csat='Not Satisfied' then profile_identity end) as notSatisfied,
    count(case when event_props_csat='' then profile_identity end) as notRated
    from clevertap.captain_bot_call_support_immutable
    group by 1
    )
    select a.*,b.totalDistinctRatedCaps,b.totalRatingGiven,b.Satisfied,b.notSatisfied,b.notRated
    from calling_data a
    left join rating_data b
    on a.yyyymmdd=b.yyyymmdd
"""
langcityservice=pd.read_sql(q,metabase_connection)

In [34]:
langcityservice.to_clipboard()

In [10]:
langV1.to_clipboard()

In [22]:
langcityservice.to_clipboard()

In [20]:
q="""
   with bot_initiated as(
  select yyyymmdd,count(distinct event_props_rider_id) as distinctCallingCaps,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  and event_props_context='bot_context__bob'
  group by 1
),
bot_csat as(
   select yyyymmdd,count(distinct event_props_rider_id) as distinctRatedCaps,
  count(event_props_rider_id) as totalRatings,
  count(case when event_props_csat='Satisfied' then event_props_rider_id end) as SatisfiedCount,
  count(case when event_props_csat='Not Satisfied' then event_props_rider_id end) as UnSatisfiedCount,
  count(case when event_props_csat='' then event_props_rider_id end) as Not_rated_count
  from  clevertap.captain_bot_csat_immutable
 where yyyymmdd between '20250206' and '20250223'
 and event_props_context='bot_context__bob'
  group by 1
)
select a.*,b.distinctRatedCaps,b.totalRatings,b.SatisfiedCount,b.UnSatisfiedCount,b.Not_rated_count
from bot_initiated a
left join bot_csat b
on a.yyyymmdd=b.yyyymmdd

"""
dod=pd.read_sql(q,metabase_connection)

DatabaseError: Execution failed on sql: 
   with bot_initiated as(
  select yyyymmdd,count(distinct event_props_rider_id) as distinctCallingCaps,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  and event_props_context='bot_context__bob'
  group by 1
),
bot_csat as(
   select yyyymmdd,count(distinct event_props_rider_id) as distinctRatedCaps,
  count(event_props_rider_id) as totalRatings,
  count(case when event_props_csat='Satisfied' then event_props_rider_id end) as SatisfiedCount,
  count(case when event_props_csat='Not Satisfied' then event_props_rider_id end) as UnSatisfiedCount,
  count(case when event_props_csat='' then event_props_rider_id end) as Not_rated_count
  from  clevertap.captain_bot_csat_immutable
 where yyyymmdd between '20250206' and '20250223'
 and event_props_context='bot_context__bob'
  group by 1
)
select a.*,b.distinctRatedCaps,b.totalRatings,b.SatisfiedCount,b.UnSati

In [29]:
dod.to_clipboard()

In [41]:
callback=pd.read_sql(q,metabase_connection)


KeyboardInterrupt: 

In [221]:
callback

NameError: name 'callback' is not defined

In [228]:
q="""
   with bot_initiated as (
        select
            yyyymmdd, count(*) as totalInitiated
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250206' and '20250224'
        and event_props_context='bot_context__bob'
        and event_props_event_name='BOT_INITIATED'
        group by 1
    ), 
    rating_captains as (
        select yyyymmdd,count(*) as totalRatingGiven
        from clevertap.captain_bot_csat_immutable
        where event_props_context='bot_context__bob'
        and yyyymmdd between '20250206' and '20250224'
        group by 1
    ),
    call_data as (
       select yyyymmdd,count(*) as totalCalls
       from clevertap.captain_bot_call_support_immutable 
       where event_props_event_name='BOT_CALL_SUPPORT'
       and yyyymmdd between '20250206' and '20250224'
       and event_props_context='bot_context__bob'
       group by 1
    )
    select a.yyyymmdd,a.totalInitiated,b.totalRatingGiven,c.totalCalls
    from bot_initiated a
    left join rating_captains b
    on b.yyyymmdd=a.yyyymmdd
    left join call_data c
    on c.yyyymmdd=a.yyyymmdd
"""

In [229]:
dod=pd.read_sql(q,metabase_connection)

In [230]:
dod.to_clipboard()

In [120]:
q="""
  select
              json_extract_scalar(event_props,'$.riderId') as rider_id, epoch
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250206' and '20250224'
        and event_props_context='bot_context__bob'
        and event_props_event_name='BOT_INITIATED'
        order by 1,2
"""
botInitiated=pd.read_sql(q,metabase_connection)

In [ ]:
"""finalTbl as(
       select a.*,b.totalRatingGiven,c.totalCalls
       from bot_initiated a
       left join rating_captains b
       on a.rider_id=b.rider_id
       left join call_data c
       on a.rider_id=c.rider_id
    ),
    finalV1 as (

        select a.*,b.performance_segment, b.consistency_segment
        from finalTbl a
        inner join  (
                        select captain_id,performance_segment,consistency_segment
                        from datasets_internal.captain_subservice_segments_v1
                        where run_date='20250224'
                       and captain_id in (select rider_id from finalTbl)  
                    ) b
        on a.rider_id=b.captain_id
    ),
    finalV2 as (
    select a.*,b.gig_segment
        from finalV1 a
        inner join  (
                        select captain_id,gig_segment
                        from reports_internal.captain_gig_appography_v4 
                        where run_date = '20250107'
                        and captain_id in (select rider_id from finalTbl)  
                    ) b
        on a.rider_id=b.captain_id
    )"""

,rider_id,totalInitiated
0,6364d86376b63052c8faa233,1
1,5dedd39c3858de33801aec92,1
2,6257cfce91957f5e0492cd49,1
3,5ce50486377155163c01c498,2
4,5d11fe6c4253186bbaff4aa3,2
...,...,...
234986,63917b156cf814781717f09e,2
234987,664afd6057b4342fcfe7f0a2,1
234988,64e2da33901b86d7ae2ee9f3,1
234989,611b5c6634b93d2c5a2ae375,4


In [ ]:
q="""
   select profile_identity as rider_id,count(profile_identity) as totalRatingGiven
        from clevertap.captain_bot_csat_immutable
        where event_props_event_name='BOT_INITIATED'
        and event_props_context='bot_context__bob'
        group by 1
"""
ratedData=pd.read_sql(q,metabase_connection)

In [186]:
q="""
  select json_extract_scalar(event_props,'$.riderId') as rider_id,epoch
       from clevertap.captain_bot_call_support_immutable 
       where event_props_event_name='BOT_CALL_SUPPORT'
       and event_props_context='bot_context__bob'
       and yyyymmdd between '20250206' and '20250224'
"""
callingData=pd.read_sql(q,metabase_connection)

In [ ]:
q="""
    select  json_extract_scalar(event_props,'$.riderId') as rider_id,epoch
        from clevertap.captain_bot_csat_immutable
        where event_props_context='bot_context__bob'
        and yyyymmdd between '20250206' and '20250224'
        order by 1,2
"""
ratingData=pd.read_sql(q,metabase_connection)

In [72]:
ratingData[ratingData.rider_id.str.contains(',')].rider_id.iloc[10]

'5d6a4622a24c96105e302fd4,61d2d53ca75746cf26e6b079'

In [126]:
ratingData['data']='Rating'

In [127]:
botInitiated['data']='Initiated'

In [140]:
finalDf=pd.concat([ratingData,botInitiated],axis=0)

In [141]:
finalDf=finalDf.sort_values(by=['rider_id','epoch']).reset_index(drop='index')

In [189]:
finalDf

,rider_id,epoch,data,rated_epoch,finalState
0,5737ca76ddbec220de03eb4b,1739509610000,Initiated,1.739544e+12,Initiated
1,5737ca76ddbec220de03eb4b,1739544067000,Initiated,1.739596e+12,Initiated
2,5737ca76ddbec220de03eb4b,1739596374000,Initiated,1.739602e+12,Initiated
3,5737ca76ddbec220de03eb4b,1739602103000,Initiated,1.739627e+12,Initiated
4,5737ca76ddbec220de03eb4b,1739626642000,Initiated,1.739630e+12,Initiated
...,...,...,...,...,...
2882737,67bc76a992db5f71999fdc06,1740417673000,Initiated,NaN,NaN
2882738,67bc78bec4512b7c68fc5f6f,1740406310000,Initiated,1.740406e+12,Rating
2882739,67bc78bec4512b7c68fc5f6f,1740406337000,Rating,NaN,NaN
2882740,67bc7f9c92db5fbb40a2c9a9,1740420362000,Initiated,NaN,NaN


In [159]:
finalDf['rated_epoch'] = finalDf.groupby(['rider_id'])['epoch'].shift(-1)

In [160]:
finalDf['finalState'] = finalDf.groupby(['rider_id'])['data'].shift(-1)

In [164]:
analysisDf=finalDf[(finalDf['data']!=finalDf['finalState'])]

In [166]:
analysisDf.reset_index(drop='index',inplace=True)

In [168]:
analysisDf=analysisDf[analysisDf.finalState=='Rating']

In [169]:
analysisDf.reset_index(drop='index',inplace=True)

In [173]:
analysisDf=analysisDf.astype({'epoch':'float'})

In [174]:
analysisDf

,rider_id,epoch,data,rated_epoch,finalState
0,573f28f39b0ffc283676f0f0,1.739867e+12,Initiated,1.739867e+12,Rating
1,573f28f59b0ffc283676fd72,1.739543e+12,Initiated,1.739543e+12,Rating
2,573f28f89b0ffc2836771015,1.739995e+12,Initiated,1.739995e+12,Rating
3,573f29019b0ffc28367728c6,1.739622e+12,Initiated,1.739622e+12,Rating
4,573f29019b0ffc28367728c6,1.739632e+12,Initiated,1.739632e+12,Rating
...,...,...,...,...,...
240487,67bc66e5925e761836ae330e,1.740414e+12,Initiated,1.740414e+12,Rating
240488,67bc672d925e76684fae4ae3,1.740406e+12,Initiated,1.740406e+12,Rating
240489,67bc672d925e76684fae4ae3,1.740408e+12,Initiated,1.740408e+12,Rating
240490,67bc74bd856a3c83e788436d,1.740414e+12,Initiated,1.740414e+12,Rating


In [175]:
analysisDf['timeTaken']=(analysisDf['rated_epoch']-analysisDf['epoch'])/1000

In [195]:
analysisDf.timeTaken.describe([0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.7,0.75,0.8,0.85,0.9,0.95,0.99]).reset_index().to_clipboard()

In [231]:
q="""
   with callingData as (
      select distinct yyyymmdd,json_extract_scalar(event_props,'$.riderId') as rider_id
       from clevertap.captain_bot_call_support_immutable 
       where event_props_event_name='BOT_CALL_SUPPORT'
       and yyyymmdd between '20250206' and '20250224'
       and event_props_context='bot_context__bob'
    ), 
    galaxy_data as (
       select a.*
       from (
            select yyyymmdd,epoch,json_extract_scalar(event_props,'$.riderId') as rider_id,json_extract_scalar(event_props,'$.disposition') as disposition,
                json_extract_scalar(event_props,'$.subDisposition') as subDisposition,
                json_extract_scalar(event_props,'$.holdCount') as holdCount,
                json_extract_scalar(event_props,'$.startTime') startTime,
                json_extract_scalar(event_props,'$.endTime') hangupTime
            from  clevertap.captain_callback_galaxy_immutable
            where yyyymmdd between '20250206' and '20250224'
                and  json_extract_scalar(event_props,'$.call_type')='IB'
                and cast(json_extract_scalar(event_props,'$.dispositionWrapTime') as decimal)<=600
                and json_extract_scalar(event_props,'$.riderId') in (select rider_id from callingData)
       ) a
       inner join callingData b
       on a.rider_id=b.rider_id
       and a.yyyymmdd=b.yyyymmdd
    )
    select * from galaxy_data

"""
galaxyData=pd.read_sql(q,metabase_connection)

In [134]:
galaxyData.dtypes

yyyymmdd          object
rider_id          object
disposition       object
subDisposition    object
holdCount         object
startTime         object
hangupTime        object
dtype: object

In [146]:
disposition=galaxyData.groupby(['disposition','subDisposition']).agg({'rider_id':'count'}).reset_index()

In [150]:
disposition=disposition.sort_values(by='rider_id',ascending=False).reset_index(drop='index')

In [154]:
disposition.loc[0:10].to_clipboard()

In [183]:
disposition.to_clipboard()

In [188]:
callingData

,rider_id,epoch
0,63e714757f3dc00a73ae9c45,1740390792000
1,640577ec71af5901d6ea880b,1740392828000
2,662132745bc5a84b6c196874,1740392167000
3,6303ad3c04cb1baf7a0dc25a,1740390511000
4,64ae7667e637fe1ebcd1c869,1740389974000
...,...,...
132666,5e01a386277dbb0268bb5d05,1738844870000
132667,5db3d23b0d30864f028585d9,1738842620000
132668,62d7e1fe0d3bae6255e46ba6,1738842492000
132669,5c52f5d04a267149c780af0f,1739678699000


In [191]:
finalDf

,rider_id,epoch,data,rated_epoch,finalState
0,5737ca76ddbec220de03eb4b,1739509610000,Initiated,1.739544e+12,Initiated
1,5737ca76ddbec220de03eb4b,1739544067000,Initiated,1.739596e+12,Initiated
2,5737ca76ddbec220de03eb4b,1739596374000,Initiated,1.739602e+12,Initiated
3,5737ca76ddbec220de03eb4b,1739602103000,Initiated,1.739627e+12,Initiated
4,5737ca76ddbec220de03eb4b,1739626642000,Initiated,1.739630e+12,Initiated
...,...,...,...,...,...
2882737,67bc76a992db5f71999fdc06,1740417673000,Initiated,NaN,NaN
2882738,67bc78bec4512b7c68fc5f6f,1740406310000,Initiated,1.740406e+12,Rating
2882739,67bc78bec4512b7c68fc5f6f,1740406337000,Rating,NaN,NaN
2882740,67bc7f9c92db5fbb40a2c9a9,1740420362000,Initiated,NaN,NaN


,rider_id,epoch,data,rated_epoch,finalState,timeTaken
6,573f29029b0ffc2836773114,1.739422e+12,Initiated,1.739422e+12,Rating,27.0
10,573f29029b0ffc2836773114,1.739424e+12,Initiated,1.739424e+12,Rating,21.0
13,573f290a9b0ffc283677470f,1.740229e+12,Initiated,1.740229e+12,Rating,27.0
21,5752989894131a58794e7489,1.739442e+12,Initiated,1.739442e+12,Rating,26.0
22,5752989894131a58794e7489,1.739521e+12,Initiated,1.739521e+12,Rating,27.0
...,...,...,...,...,...,...
240469,67babdfe5c8f83168cda55e3,1.740344e+12,Initiated,1.740344e+12,Rating,29.0
240476,67bb0e1c925e763d1f3c11c2,1.740358e+12,Initiated,1.740358e+12,Rating,25.0
240485,67bc2c36c51d5d04979475e3,1.740397e+12,Initiated,1.740397e+12,Rating,20.0
240487,67bc66e5925e761836ae330e,1.740414e+12,Initiated,1.740414e+12,Rating,29.0


In [237]:
galaxyData['data']="Calling"


In [265]:
analysisDf2=pd.concat([analysisDf,galaxyData],axis=0)

In [266]:
analysisDf2.sort_values(by=['rider_id','epoch'],inplace=True)

In [267]:
analysisDf2.reset_index(drop='index',inplace=True)

In [268]:
analysisDf2['calling_time'] = analysisDf2.groupby(['rider_id'])['epoch'].shift(-1)

In [269]:
analysisDf2['finalState2']=analysisDf2.groupby(['rider_id'])['data'].shift(-1)

In [270]:
analysisDf2['finalSubDis']=analysisDf2.groupby(['rider_id'])['subDisposition'].shift(-1)

In [271]:
analysisDf2['finalDispo']=analysisDf2.groupby(['rider_id'])['disposition'].shift(-1)

In [272]:
analysisDf2['finalStartTime']=analysisDf2.groupby(['rider_id'])['startTime'].shift(-1)

In [273]:
analysisDf2['finalhanguptime']=analysisDf2.groupby(['rider_id'])['hangupTime'].shift(-1)

In [277]:
finalDf=analysisDf2[(analysisDf2.data=='Initiated') & (analysisDf2.finalState=='Rating') & (analysisDf2.finalState2=='Calling')]

In [278]:
finalDf=finalDf[['rider_id','data','finalState','finalState2','epoch','rated_epoch','calling_time','finalDispo','finalSubDis','finalStartTime','finalhanguptime']]

In [282]:
finalDf['finalStartTime']=pd.to_datetime(finalDf.finalStartTime)

In [284]:
finalDf['finalhanguptime']=pd.to_datetime(finalDf.finalhanguptime)

In [292]:
finalDf['timetorate']=(finalDf.rated_epoch-finalDf.epoch)/1000
finalDf['timetocall']=(finalDf.calling_time-finalDf.rated_epoch)/1000

In [294]:
finalDf['timeofCall']=pd.Timedelta(finalDf.finalhanguptime- finalDf.finalStartTime).total_seconds()

ValueError: Value must be Timedelta, string, integer, float, timedelta or convertible, not Series

In [ ]:
def timeDiff(df):
    pd.Timedelta(finalDf.finalhanguptime- finalDf.finalStartTime).total_seconds()


In [305]:
finalDf['talkingTime'] = (finalDf.finalhanguptime- finalDf.finalStartTime) /pd.Timedelta(seconds=1)

In [308]:
finalDf.timetorate.describe([0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7]).to_clipboard()

In [335]:
len(finalDf[finalDf['timetorate']<119])

78232

In [336]:
len(analysisDf[analysisDf['timeTaken']<119])

203327

In [383]:
query="""
    with byob_confirmed as (
    select distinct
      min(yyyymmdd) over(partition by eventprops__eventprops__userid order by yyyymmdd) as yyyymmdd,
      eventprops__eventprops__userid as captain_id,eventprops__eventprops__city city
    from
      canonical.iceberg_production_data_raw_clevertap_captain_clevertap_captain_events_v2_eventname_byob_screen_immutable
    where eventprops__eventprops__action='plan_confirmed'
    and yyyymmdd between '20250124' and '20250223'
),
byob_test_caps as(
  select a.yyyymmdd,a.captain_id,date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d')) as cohot,
      yyyymmdd as post_start_date,city,
      date_format(date_add('day',date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d')),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as post_end_date,
      date_format(date_add('day',-1,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_end_date,
      date_format(date_add('day',-(date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d'))+1),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_start_date
  from byob_confirmed a
),
orders as(
 select yyyymmdd,captain_id,count(distinct order_id) as numOrders
 from orders.order_logs_fact
 where order_status='dropped'
 and yyyymmdd between '20241223' and '20250224'
 and captain_id in (select captain_id from byob_test_caps)
 group by 1,2
),
cm as (
  select yyyymmdd,captain_id,sum(total_login_hours) as login_hrs
  from datasets.captain_metrics as cm
  where yyyymmdd between '20241223' and '20250224'
  and captain_id in (select captain_id from byob_test_caps)
  group by 1,2
),
totalEarning as( select yyyymmdd,rider_id as captain_id,
    coalesce(sum(case when service = 'Link' then captain_earnings end),0) as earning
    from captain.captain_transactions_snapshot
            where 1=1
            and (transaction_type = 'orders' or transaction_type='incentives')
            and yyyymmdd between '20241223' and '20250224'
            and rider_id in (select captain_id from byob_test_caps)
    group by 1,2
),
metric_orders as (
   select a.yyyymmdd,a.captain_id,a.numOrders,b.login_hrs
   from orders a
   left join cm b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
metric_order_earning as(
  select a.yyyymmdd,a.captain_id,a.numOrders,a.login_hrs,b.earning
   from metric_orders a
   left join totalEarning b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
finalTbl as (
   select a.yyyymmdd,a.city,a.cohot,a.captain_id,
        count(distinct case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.yyyymmdd end) as pre_netDays,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.numOrders else 0 end) as pre_numOrders,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.login_hrs else 0 end) as pre_login_hrs,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.earning else 0 end) as pre_total_earnings,
        count(distinct case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.yyyymmdd end) as post_netDays,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.numOrders else 0  end) as post_numOrders,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.login_hrs else 0 end) as post_login_hrs,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.earning else 0 end) as post_total_earnings
    from byob_test_caps a
    inner join metric_order_earning b
    on a.captain_id=b.captain_id
    group by 1,2,3,4
),
analysisTbl as (
    select yyyymmdd,cohot,city,count(distinct captain_id) as totalCaptains,sum(pre_numOrders) as preOrders,sum(pre_netDays) as preNetDays,sum(pre_login_hrs) as preLoginHrs,
    sum(pre_total_earnings) as preTotalEarnings,
    sum(post_total_earnings) as postTotalEarnings,
    sum(post_numOrders) as postOrders,sum(post_netDays) as postNetDays,
    sum(post_login_hrs) as postLoginHrs
    from finalTbl
    group by 1,2,3
)
select * , (RPR_Post - RPR_Pre) as RPR_Change, (Net_days_Post - Net_days_Pre) as Net_days_Change, (LH_Post - LH_Pre) as LH_Change,(Earnings_post - Earnings_pre) as Earnings_change
from (
select yyyymmdd,cohot,city,
totalCaptains as Total_Captains,
(preOrders*1.0/totalCaptains) as RPR_Pre,  (preNetDays*1.0/totalCaptains) as Net_days_Pre, (preLoginHrs*1.0 / totalCaptains) as LH_Pre,(preTotalEarnings*1.0/totalCaptains) as Earnings_pre,
(postOrders*1.0/totalCaptains) as RPR_Post,  (postNetDays*1.0/totalCaptains) as Net_days_Post, (postLoginHrs*1.0 / totalCaptains) as LH_Post,(postTotalEarnings*1.0/totalCaptains) as Earnings_post
from analysisTbl
)
"""
hypothesis=pd.read_sql(query,metabase_connection)

In [384]:
hypothesis.sort_values(by='cohot', ascending=False).to_clipboard()

In [ ]:
q="""
  with  post as ( 
select 
count(json_extract_scalar(event_props,'$.userId')) as PosttotalUser
from clevertap.captain_supportitem_clicked_immutable
where yyyymmdd between '20250206' and '20250224'
and event_props_mode = 'Link'
and lower(event_props_source) like 'helpfaq'
),
 pre as (
   select count(json_extract_scalar(event_props,'$.userId')) as PretotalUser
from clevertap.captain_supportitem_clicked_immutable
where yyyymmdd between '20250118' and '20250205'
and event_props_mode = 'Link'
)
select b.PretotalUser,a.PosttotalUser
from post a
cross join pre b
"""
prepostFaq=pd.read_sql(q,metabase_connection)

In [450]:
q="""
select *
from raw.kafka_domain_captain_subscription_rules_immutable rules
where yyyymmdd='20250203'
limit 10
"""

In [451]:
df=pd.read_sql(q,metabase_connection)

In [452]:
df

,id,hhmmss,event_epoch,epoch,updated_quarter_hour,updated_epoch,updated_hhmmss,updated_yyyymmdd,hh,metadata,...,createdBy,createdOn,excludeIncentiveTypes,exclusiveOfTax,validity,isAutoAssignedToHH,updatedAt,version,source,data


In [448]:
df

,id,hhmmss,event_epoch,epoch,updated_quarter_hour,updated_epoch,updated_hhmmss,updated_yyyymmdd,hh,metadata,...,createdBy,createdOn,excludeIncentiveTypes,exclusiveOfTax,validity,isAutoAssignedToHH,updatedAt,version,source,data


In [35]:
day_week_month = 'week'
start_ymd = '20241001'
end_ymd = '20250228'
perf_cut = 1
cons_cut = 1
df_city_trends = pd.read_sql(
    f'''
    with base as (
        select 
            city,
            captainid, 
            date_format(
                date_parse(yyyymmdd,'%Y%m%d'), 
                case
                    when '{day_week_month}' = 'day' then '%Y-%m-%d'
                    when '{day_week_month}' = 'week' then '%Y-%v'
                    when '{day_week_month}' = 'month' then '%Y-%m'
                end
                ) as week_number,
            -- yyyymmdd,
            count(distinct case when net_orders>0 then yyyymmdd end) as net_days,
            sum(net_orders) as net_orders, 
            sum(order_earning) as order_earning, 
            sum(accepted_pings) as accepted_pings
        from datasets.captain_svo_daily_kpi
        where yyyymmdd between '{start_ymd}' and '{end_ymd}'
        and net_orders > 0
        and lower(ordertype) like '%cab%'
        and servicedetailid in (select service_detail_id from datasets.service_level_mapping_qc where lower(service_category) like '%cab%' and city_display_name in ('chennai', 'delhi', 'mumbai', 'pune', 'bangalore', 'hyderabad', 'jaipur'))
        group by 1,2,3
    )

    , pc_v2 as (
        select 
            -- distinct 
            captain_id, 
            -- run_date yyyymmdd,
            date_format(
                date_parse(run_date,'%Y%m%d'), 
                case
                    when '{day_week_month}' = 'day' then '%Y-%m-%d'
                    when '{day_week_month}' = 'week' then '%Y-%v'
                    when '{day_week_month}' = 'month' then '%Y-%m'
                end
                ) as week_number, 
            max(case when performance_segment = 'UHP' then 4 when performance_segment = 'HP' then 3 when performance_segment= 'MP' then 2 when performance_segment = 'LP' then 1 else 0 end) performance,  
            max(case when lower(consistency_segment) = 'daily' then 4 when lower(consistency_segment) = 'inter' then 3 when lower(consistency_segment) = 'intra' then 2 when lower(consistency_segment) = 'sporadic' then 1 else -1 end) consistency
        from datasets_internal.captain_subservice_segments_v1
        where run_date between '{start_ymd}' and '{end_ymd}'
        and day_of_week(date_parse(run_date, '%Y%m%d')) = 7
        and captain_id in (select distinct captainid from base)
        and lower(service_mode) = 'all'
        and lower(service) like '%cab%'
        group by 1,2
    )

    select city, 
        case when {perf_cut} != 1 then 'NA' else performance end performance,
        case when {cons_cut} != 1 then 'NA' else consistency end consistency,  
        week_number, 
        avg(net_days) as avg_net_days, 
        count(distinct captainid) as weekly_net_caps,
        avg(net_orders) as net_orders,
        avg(order_earning) as order_earning,
        avg(dapr) as dapr
    from (
        select distinct
            'T1' as city, 
            captainid,
            case when performance = 1 then 'LP' when performance = 2 then 'MP' when performance = 3 then 'HP' when performance = 4 then 'UHP' else 'NA' end as performance,
            case when consistency = 1 then 'Sporadic' when consistency = 2 then 'Inter' when consistency = 3 then 'Intra' when consistency = 4 then 'Daily' else 'NA' end as consistency, 
            a.week_number, 
            a.net_days as net_days, 
            a.net_orders/cast(net_days as double) as net_orders, 
            order_earning/cast(net_days as double) as order_earning, 
            case when cast(accepted_pings as int)=0 then 0
            else  net_orders/cast(accepted_pings as double) 
            end as dapr
        from base a
        left join pc_v2 b on a.captainid = b.captain_id and a.week_number = b.week_number
        -- group by 1,2,3,4,5
    )
    group by 1,2,3,4

    union all

    select 
        city, 
        case when {perf_cut} != 1 then 'NA' else performance end performance,
        case when {cons_cut} != 1 then 'NA' else consistency end consistency, 
        week_number, 
        avg(net_days) as avg_net_days, 
        count(distinct captainid) as weekly_net_caps, 
        avg(net_orders) as net_orders,
        avg(order_earning) as order_earning,
        avg(dapr) as dapr
    from (
        select distinct
            city, 
            captainid,
            case when performance = 1 then 'LP' when performance = 2 then 'MP' when performance = 3 then 'HP' when performance = 4 then 'UHP' else 'NA' end as performance,
            case when consistency = 1 then 'Sporadic' when consistency = 2 then 'Inter' when consistency = 3 then 'Intra' when consistency = 4 then 'Daily' else 'NA' end as consistency, 
            a.week_number, 
            a.net_days,
            a.net_orders/cast(net_days as double) as net_orders, 
            a.order_earning/cast(net_days as double) as order_earning, 
            case when cast(accepted_pings as int)=0 then 0
            else  net_orders/cast(accepted_pings as double) 
            end as dapr
        from base a
        left join pc_v2 b on a.captainid = b.captain_id and a.week_number = b.week_number
        -- group by 1,2,3,4,5
    )
    group by 1,2,3,4
    order by 1,2,3,4
    ''', metabase_connection)

In [37]:
df_city_trends.to_clipboard()

In [11]:
df_city_trends

,city,performance,consistency,week_number,avg_net_days,weekly_net_caps,accepted_pings,net_orders,order_earning,dapr
0,T1,HP,Daily,2024-10,24.258333,119,270.783333,7.861661,1339.197005,0.714761
1,T1,HP,Daily,2024-11,23.183857,223,287.793722,8.820090,1560.243161,0.718175
2,T1,HP,Daily,2024-12,24.627049,488,277.032787,7.889096,1520.871715,0.711391
3,T1,HP,Daily,2025-01,24.635458,502,291.141434,8.293088,1671.952828,0.713244
4,T1,HP,Daily,2025-02,22.325073,685,285.969388,8.881104,1847.193451,0.707169
...,...,...,...,...,...,...,...,...,...,...
565,pune,NA,NA,2024-12,1.827853,517,4.526112,1.857668,425.292496,0.847084
566,pune,NA,NA,2025-01,1.922414,928,5.703664,1.956012,443.671339,0.81312
567,pune,NA,NA,2025-02,2.293961,1126,6.214920,1.774300,451.521554,0.810568
568,pune,UHP,Daily,2025-01,29.000000,1,334.000000,7.344828,1534.931034,0.637725
